<a href="https://colab.research.google.com/github/AmirhosseinSoltan/AmirhosseinSoltan/blob/main/face_coverings_detection_yolo_11_map50_0_93.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the ultralytics package
%pip install ultralytics

# Import and check the setup (Verifies GPU/CPU and software versions)
import ultralytics
ultralytics.checks()

Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.5/112.6 GB disk)


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')

In [ ]:
import torch
# Check device availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Current device: {device}")

CUDA available: True
Number of GPUs: 1
Current device: cuda


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("ashokkumarmalineni/face-coveringsaccessories-detection")

# print("Path to dataset files:", path)

In [ ]:
import os
# Check data structure
print("\nChecking data availability...")
# base_path = '/kaggle/input/face-coveringsaccessories-detection'
# base_path = path
base_path = "/content"

if os.path.exists(base_path):
    print("Directory found:")
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            print(f"{item}: {len(os.listdir(item_path))} items")
        else:
            print(f"{item}")
else:
    print("Directory not found!")
    exit()

# Check data.yaml
cfg = os.path.join(base_path, 'data.yaml')
try:
    with open(cfg, 'r') as f:
        data_config = f.read()
    print("\ndata.yaml found:")
    print(data_config)
except FileNotFoundError:
    print("data.yaml not found!")
    exit()


Checking data availability...
Directory found:
.config: 10 items
yolo11n.pt
facial_occlusion: 1 items
data.yaml not found!


In [ ]:
from ultralytics import YOLO
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import pandas as pd


# Load model
print("\nLoading YOLO model...")
model = YOLO('yolo11n.pt')

# Training parameters
print(f"Starting training on {device.upper()}...")
results = model.train(
    data=cfg,
    epochs=50,
    imgsz=640,
    batch=16 if device == 'cuda' else 8,  # Adjust batch size based on device
    device=device,
    patience=15,
    project='facial_occlusion',
    name='yolov11_ft',
    verbose=True,
    workers=4 if device == 'cuda' else 2  # Adjust workers based on device
)

print("Training completed!")



Loading YOLO model...
Starting training on CUDA...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11_ft2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

RuntimeError: Dataset '/content/data.yaml' error ❌ '/content/data.yaml' does not exist

# Display training results and metrics

In [ ]:


print("\nTraining Results Analysis:")
results_dir = '/content/facial_occlusion/yolov11_ft'

# if os.path.exists(results_dir):
#   print(f"\nContents of results directory:")
# for item in os.listdir(results_dir):
#     print(f"{item}")

# Read and display results.csv if exists
results_csv = os.path.join(results_dir, 'results.csv')
if os.path.exists(results_csv):
    print(f"\nTraining metrics from results.csv:")
    df = pd.read_csv(results_csv)
    print(df.tail())  # Show last few rows

    # Display key metrics
    if not df.empty:
        print(f"\nFinal training metrics:")
        print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.3f}")
        print(f"Best precision: {df['metrics/precision(B)'].max():.3f}")
        print(f"Best recall: {df['metrics/recall(B)'].max():.3f}")

# Show training batch images
batch_images = glob.glob(os.path.join(results_dir, 'train_batch*.jpg'))
if batch_images:
    print(f"\nTraining batch images:")
    fig, axes = plt.subplots(1, min(3, len(batch_images)), figsize=(15, 5))

    if min(3, len(batch_images)) == 1:
        axes = [axes]

    for i, img_path in enumerate(batch_images[:3]):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f'Train Batch {i}')
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Model validation


In [ ]:

print("Running validation...")
metrics = model.val()
print("Validation metrics:")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP75: {metrics.box.map75:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# # Check for test data and make predictions
# test_path = '/kaggle/input/face-coveringsaccessories-detection/test/images'
# if os.path.exists(test_path) and os.listdir(test_path):
#     print("Running predictions on test data...")
#     test_results = model.predict(
#         source=test_path,
#         save=True,
#         conf=0.25,
#         save_txt=True,
#         project='test_predictions',
#         device=device
#     )
#     print("Test predictions completed!")
# else:
#     print("Test folder not found or empty, making predictions on validation data...")

# Get 10 random images from validation set
valid_path = os.path.join(base_path, "valid/images")
valid_images = os.listdir(valid_path)[:10]  # Take first 10 images

if valid_images:
    print(f"Making predictions on {len(valid_images)} validation images...")
    valid_results = model.predict(
        source=[os.path.join(valid_path, img) for img in valid_images],
        save=True,
        conf=0.25,
        save_txt=True,
        project='./preds',
        device=device
    )

    # Visualize predictions
    print("\nVisualizing predictions...")
    prediction_dir = './preds'
    if os.path.exists(prediction_dir):
        predicted_images = glob.glob(os.path.join(prediction_dir, '*.jpg'))[:6]  # Show first 6 images

        if predicted_images:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.ravel()

            for i, img_path in enumerate(predicted_images[:6]):
                img = mpimg.imread(img_path)
                axes[i].imshow(img)
                axes[i].set_title(f'Prediction {i+1}')
                axes[i].axis('off')

            plt.tight_layout()
            plt.show()
            print("Prediction visualization completed!")
else:
    print("No validation images found!")


# Show validation results
val_images = glob.glob(os.path.join(results_dir, 'val_batch*.jpg'))
if val_images:
    print(f"\nValidation results:")
    for img_path in val_images[:2]:  # Show first 2 validation images
        img = mpimg.imread(img_path)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title('Validation Results')
        plt.axis('off')
        plt.show()
else:
  print("Results directory not found!")

print("\nAll operations completed successfully!")
print(f"Model saved at: ./facial_occlusion/weights/best.pt")